# Write GenBank files for plasmid log
I have designed HA expression constructs linked to 16-nucleotide barcodes. We will clone these into the pHH21 derivative (Bloom lab plasmid ID 2851). I want to save annotated expected sequence information along as a GenBank file. I can use this file to submit to the Bloom lab plasmid log. 

Author: Caroline Kikawa

In [18]:
# Import relevant packages
import os
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqFeature import SeqFeature, FeatureLocation
import pandas as pd
import glob
import re

import warnings
from Bio import BiopythonWarning
warnings.simplefilter("ignore", BiopythonWarning) # Silence warnings about long plasmid names


# ID input and output
datadir = '../data'
resultsdir = '../results'
os.makedirs(datadir, exist_ok=True)
os.makedirs(resultsdir, exist_ok=True)

genbankdir = os.path.join('../plasmids')
os.makedirs(genbankdir, exist_ok=True)

ordersheetsdir = os.path.join(resultsdir, 'ordersheets')

First, annotate the library ID file with a column called 'clone.' Write each unique clone identifier in this column. Then, put all Primordium files in the 'data' directory. If a given clone would have 2 sequences for any reason, choose only 1 to upload to the data directory. 

If a given library sequence doesn't have 3 (or any) chosen clones, a genbank file with barcode ID 'NNNNNNNNNNNNNNNN' will be generated. 

In [19]:
# Get inserts from ordersheets
inserts_df = pd.concat([
    pd.read_csv(os.path.join(ordersheetsdir, 'h1_inserts.csv')),
    pd.read_csv(os.path.join(ordersheetsdir, 'h3_inserts.csv')),
    pd.read_csv(os.path.join(ordersheetsdir, 'vaccine_inserts.csv')),
    pd.read_csv(os.path.join(ordersheetsdir, 'lisboa_inserts.csv')),
]).reset_index(drop=True)

inserts_df = inserts_df.assign(
    subtype = lambda x: x['name'].str.split('_').str[0]
)

# For each strain, trim for HA nucleotide sequence, HA protein sequence (NO CHIMERIC SEQUENCE AT ALL!!)
# Get barcode
seq_list = []

for n in inserts_df.name.unique():
    seq = inserts_df.query(f'name == "{n}"')['sequence'].iloc[0]
    if inserts_df.query(f'name == "{n}"').subtype.iloc[0] == 'H1N1':
        ha_ecto_seq = seq[29:-180]
    elif inserts_df.query(f'name == "{n}"').subtype.iloc[0] == 'H3N2':
        ha_ecto_seq = seq[26:-189]        
    barcode = seq[-36:-20]
    seq_list.append([n, ha_ecto_seq, barcode])

 

# Merge trimmed sequence
inserts_df = inserts_df.merge(pd.DataFrame(seq_list, columns = ['name', 'trimmed_ha_ecto_sequence', 'barcode']))
# Get protein HA ectodomain sequence
inserts_df['trimmed_ha_ecto_sequence_protein'] = inserts_df['trimmed_ha_ecto_sequence'].apply(lambda x: str(Seq(x).translate(to_stop=True)))
# Get protein HA ectodomain sequence
inserts_df['plasmid'] = 'pHH_' + (inserts_df['strain']
    .str.replace('/','-')
    .str.replace('(','')
    .str.replace(')','')
    .str.replace("'","")
                                  
                                 ) + '_HA_WSN-flank_' + inserts_df['name'].str.split('_').str[-1]
# Strain number for ordering
inserts_df['number'] = inserts_df['name'].str.split('_').str[1].astype(int)
# Reorder
inserts_df = inserts_df.sort_values(by=['subtype', 'number'])
inserts_df = inserts_df.drop(columns='number')

# Add plate ID
plate_ID_list = []
i=1
for plate in ['plate11']:
    for row in ['A','B','C','D','E','F','G','H']:
        for col in list(range(1,13)):
            col = str(col)
            well = (''.join([row,col]))
            plate_ID = ('-'.join([plate,well]))
            plate_ID_list.append(plate_ID)
            
            # add to counter, we have 78 H1 plasmids
            i+=1
            if i >= 79:
                break  
        if i >= 79:
            break 

i=1
for plate in ['plate9','plate10']:
    for row in ['A','B','C','D','E','F','G','H']:
        for col in list(range(1,13)):
            col = str(col)
            well = (''.join([row,col]))
            plate_ID = ('-'.join([plate,well]))
            plate_ID_list.append(plate_ID)
            
            # add to counter, we have 154 H3 plasmids
            i+=1
            if i >= 155:
                break  
        if i >= 155:
            break 
inserts_df['plate_ID'] = plate_ID_list

# print forbidden characters if present
forbidden_chars = r'[<>:"/\\|?*\x00-\x1f]'
mask = inserts_df['plasmid'].str.contains(forbidden_chars, regex=True, na=False)
invalid = inserts_df[mask]['plasmid']

if invalid.empty:
    print('Plasmid names pass checks!')
else:
    print(invalid)
    raise ValueError(f"Forbidden filename characters found in {len(invalid)} plasmid name(s).")

Plasmid names pass checks!


## A few plasmids have to be removed because Azenta lost the preps
There are a few H3 plasmids we don't have Glycerol stocks of because Azenta (the company that miniprepped and made glycerol stocks for us) did not send them. Azenta claimed they had been shipped, and so didn't have another stock to re-prep a replacement glycerol stock from. 

In [20]:
# Define glycerol stocks to drop
stocks_to_drop = [
    'plate9-C7',
    'plate9-F6',
    'plate10-A7',
    'plate10-B4'
]

stocks_to_drop_df = inserts_df[inserts_df['plate_ID'].isin(stocks_to_drop)]
final_inserts_df = inserts_df[~inserts_df['plate_ID'].isin(stocks_to_drop)]

print('dropping the following stocks...')
stocks_to_drop_df

dropping the following stocks...


,strain,genbank,name,sequence,subtype,trimmed_ha_ecto_sequence,barcode,trimmed_ha_ecto_sequence_protein,plasmid,plate_ID
106,A/Busan/461/2025_H3N2,PV507425_K205R_K462R,H3N2_16_bc1,catttgtagctacagatgcagacacaCAAAAAATACCTGGAAATGA...,H3N2,CAAAAAATACCTGGAAATGACAATAGCACGGCAACGCTGTGCCTTG...,ttactacactttgcaa,QKIPGNDNSTATLCLGHHAVPNGTIVKTITNDRIEVTNATELVQNS...,pHH_A-Busan-461-2025_H3N2_HA_WSN-flank_bc1,plate9-C7
141,A/New_York/191/2024_H3N2,PQ076902,H3N2_33_bc2,catttgtagctacagatgcagacacaCAAAAAATACCTGGAAATGA...,H3N2,CAAAAAATACCTGGAAATGACAATAGCACGGCAACGCTGTGCCTTG...,taggcatcgctgtgtc,QKIPGNDNSTATLCLGHHAVPNGTIVKTITNDRIEVTNATELVQNS...,pHH_A-New_York-191-2024_H3N2_HA_WSN-flank_bc2,plate9-F6
178,A/Texas/15550/2024_H3N2,PV277098,H3N2_52_bc1,catttgtagctacagatgcagacacaCAAAAAATACCTGGAAATGA...,H3N2,CAAAAAATACCTGGAAATGACAATAGCACGGCAACGCTGTGCCTTG...,gcttgtcgcaaacagc,QKIPGNDNSTATLCLGHHAVPNGTIVKTITNDRIEVTNATELVQNS...,pHH_A-Texas-15550-2024_H3N2_HA_WSN-flank_bc1,plate10-A7
187,A/Ghana/FS-25-0256/2025_H3N2,PP279041_I41T_V128I_K205R,H3N2_56_bc2,catttgtagctacagatgcagacacaCAAAAAATACCTGGAAATGA...,H3N2,CAAAAAATACCTGGAAATGACAATAGCACGGCAACGCTGTGCCTTG...,gcctaagcgtgggcgg,QKIPGNDNSTATLCLGHHAVPNGTTVKTITNDRIEVTNATELVQNS...,pHH_A-Ghana-FS-25-0256-2025_H3N2_HA_WSN-flank_bc2,plate10-B4


Define plasmid backbone sequence that the variable fragments should combined with

In [21]:
pHH21_derivative_backbone_upstream = 'actcttcctttttcaatattattgaagcatttatcagggttattgtctcatgagcggatacatatttgaatgtatttagaaaaataaacaaaagagtttgtagaaacgcaaaaaggccatccgtcaggatggccttctgcttaatttgatgcctggcagtttatggcgggcgtcctgcccgccaccctccgggccgttgcttcgcaacgttcaaatccgctcccggcggatttgtcctactcaggagagcgttcaccgacaaacaacagataaaacgaaaggcccagtctttcgactgagcctttcgttttatttgatgcctggcagttccctactctcgcatggggagaccccacactaccatcggcgctacggcgtttcacttctgagttcggcatggggtcaggtgggaccaccgcgctactgccgccaggcaaattctgttttatcagaccgcttctgcgttctgatttaatctgtatcaggctgaaaatttttttGcGGccgccaaaacagccaagctagcggccgatccccaaaaaaaaaaaaaaaaaaagagtccagagtggccccgccgctccgcgccggggggggggggggggggggacactttcggacatctggtcgacctccagcatcgggggaaaaaaaaaaacaaagtgtcgcccggagtactggtcgacctccgaagttgggggggagcaaaagcaggggaaaataaaaacaaccaaa'
pHH21_derivative_backbone_downstream = 'agatcggaagagcgtcgtgtagggaaagagtgtgcggccgctatctactcaactgtcgccagttcactggtgctttaggtctccctgggggcaatcagtttctggatgtgttctaatgggtctttgcagtgcagaatatgcatctgagattaggatttcagaaatataaggaaaaacacccttgtttctactaataacccggcggcccaaaatgccgactcggagcgaaagatatacctcccccggggccgggaggtcgcgtcaccgaccacgccgccggcccaggcgacgcgcgacacggacacctgtccccaaaaacgccaccatcgcagccacacacggagcgcccggggccctctggtcaaccccaggacacacgcgggagcagcgccgggccggggacgccctcccggccgcccgtgccacacgcagggggccggcccgtgtctccagagcgggagccggaagcattttcggccggcccctcctacgaccgggacacacgagggaccgaaggccggccaggcgcgacctctcgggccgcacgcgcgctcagggagcgctctccgactccgcacggggactcgccagaaaggatcgtgatctgcattaatgaatcaggggataacgcaggaaagaacatgtgagcaaaaggccagcaaaaggccaggaaccgtaaaaaggccgcgttgctggcgtttttccataggctccgcccccctgacgagcatcacaaaaatcgacgctcaagtcagaggtggcgaaacccgacaggactataaagataccaggcgtttccccctggaagctccctcgtgcgctctcctgttccgaccctgccgcttaccggatacctgtccgcctttctcccttcgggaagcgtggcgctttctcatagctcacgctgtaggtatctcagttcggtgtaggtcgttcgctccaagctgggctgtgtgcacgaaccccccgttcagcccgaccgctgcgccttatccggtaactatcgtcttgagtccaacccggtaagacacgacttatcgccactggcagcagccactggtaacaggattagcagagcgaggtatgtaggcggtgctacagagttcttgaagtggtggcctaactacggctacactagaagaacagtatttggtatctgcgctctgctgaagccagttaccttcggaaaaagagttggtagctcttgatccggcaaacaaaccaccgctggtagcggtggtttttttgtttgcaagcagcagattacgcgcagaaaaaaaggatctcaagaagatcctttgatcttttctacggggtctgacgctcagtggaacgaaaactcacgttaagggattttggtcatgagattatcaaaaaggatcttcacctagatccttttaaattaaaaatgaagttttaaatcaatctaaagtatatatgagtaaacttggtctgacagttaccaatgcttaatcagtgaggcacctatctcagcgatctgtctatttcgttcatccatagttgcctgactccccgtcgtgtagataactacgatacgggagggcttaccatctggccccagtgctgcaatgataccgcgagacccacgctcaccggctccagatttatcagcaataaaccagccagccggaagggccgagcgcagaagtggtcctgcaactttatccgcctccatccagtctattaattgttgccgggaagctagagtaagtagttcgccagttaatagtttgcgcaacgttgttgccattgctacaggcatcgtggtgtcacgctcgtcgtttggtatggcttcattcagctccggttcccaacgatcaaggcgagttacatgatcccccatgttgtgcaaaaaagcggttagctccttcggtcctccgatcgttgtcagaagtaagttggccgcagtgttatcactcatggttatggcagcactgcataattctcttactgtcatgccatccgtaagatgcttttctgtgactggtgagtactcaaccaagtcattctgagaatagtgtatgcggcgaccgagttgctcttgcccggcgtcaacacgggataataccgcgccacatagcagaactttaaaagtgctcatcattggaaaacgttcttcggggcgaaaactctcaaggatcttaccgctgttgagatccagttcgatgtaacccactcgtgcacccaactgatcttcagcatcttttactttcaccagcgtttctgggtgagcaaaaacaggaaggcaaaatgccgcaaaaaagggaataagggcgacacggaaatgttgaatactcat'

Now define a function to make the plasmid maps. It should be able to add different upstream and downstream sequence depending on the HA subtype, and should annotate it with helpful things like noting the part of the HA ectodomain (which is most of it, but differs slightly between subtypes) that will vary between sequences, where the barcode is, where the Illumina Read1 sequence is, etc. 

In [22]:
def write_genbank(
    strain_name,
    accession,
    plasmid_name,
    subtype, 
    ectodomain,
    seqneut_library = 'NA',
    barcode='NNNNNNNNNNNNNNNN',
    constant_plasmid_sequence_upstream=pHH21_derivative_backbone_upstream,
    constant_plasmid_sequence_downstream=pHH21_derivative_backbone_downstream):

    # Depending on the subtype, the upstream signal peptide and downstream endododomain and CT domain will vary
    if subtype=='H3N2':
        signal_peptide_nc = 'atgaaggcaaaactactggtcctgttatatgcatttgtagctacagatgcagacaca' # first 19 aa
        endodomain_nc = 'atcaagggagttgagctgaagtcaggatacaaagattggatcctatggatttcctttgccATGtcTtgCttCCtActGtgCgtAgcACtACtAggCttTatTatgtgggcGtgTcaGaaA'
        c_term_nc = 'ggCtcCCtAcaAtgTCgGatTtgTatTTAATAG'
    elif subtype=='H1N1':
        signal_peptide_nc = 'atgaaggcaaaactactggtcctgttatatgcatttgtagctacagatgcagacacaata'
        endodomain_nc = 'aaattggaatcaatgggagtgtatcagattctggcgatatattctacagtggcaagctccttagtactgctagtttctttaggagcgattagcttttggatgtgctccaacggctccctacaatgtcggatttgtatttaatag'
        c_term_nc = '' # included in endo sequence above

    # Build the expected plasmid sequence
    plasmid_sequence = (
        constant_plasmid_sequence_upstream +
        signal_peptide_nc + 
        ectodomain +
        endodomain_nc + 
        c_term_nc + 
        barcode +
        constant_plasmid_sequence_downstream
    )

    definition = (
        f"This pHH plasmid contains the HA ectodomain sequence for a {subtype} variant {strain_name}. " +
        f"Signal peptide and 3'NCR from WSN, ectodomain from {strain_name} HA with accession {accession}, and last 46 aa recoded WSN transmembrane and c-terminal domain. " +
        f"With duplicated 5' packaging signals from WSN with a single stop codon in the duplicated packaging signal, with the barcode {barcode}. " +
        f"This construct was originally designed for the {seqneut_library} library. " +
        "This plasmid was cloned and sequence confirmed by Caroline Kikawa"
    )

    # Write sequence features
    f1 = SeqFeature(FeatureLocation(534, 700, -1), type="terminator", qualifiers = {'label': 'mouse PolI terminator'})
    f2 = SeqFeature(FeatureLocation(700, 712, -1), type="misc_feature", qualifiers = {'label': 'U12'})
    f3 = SeqFeature(FeatureLocation(700, 732, -1), type="misc_feature", qualifiers = {'label': "3' NCR"})
    
    if subtype=='H3N2':
        f4 = SeqFeature(FeatureLocation(732, 789, +1), type="misc_feature", qualifiers = {'label': 'WSN first 19 aa'})
        f5 = SeqFeature(FeatureLocation(789, 2292, +1), type="misc_feature", qualifiers = {'label': f'HA gene from {strain_name}'})
        f6 = SeqFeature(FeatureLocation(2292, 2412, +1), type="misc_feature", qualifiers = {'label': 'consensus H3 endodomain'})
        f8 = SeqFeature(FeatureLocation(2412, 2445, +1), type="misc_feature", qualifiers = {'label': 'WSN recoded CT'})
        f9 = SeqFeature(FeatureLocation(2445, 2461, +1), type="misc_feature", qualifiers = {'label': 'barcode'})
        f10 = SeqFeature(FeatureLocation(2461, 2494, +1), type="misc_feature", qualifiers = {'label': 'Illumina Read1'})
        f11 = SeqFeature(FeatureLocation(2503, 2608, +1), type="misc_feature", qualifiers = {'label': 'WSN packaging signal'})
        f12 = SeqFeature(FeatureLocation(2608, 2653, -1), type="misc_feature", qualifiers = {'label': "5' NCR"})
        f13 = SeqFeature(FeatureLocation(2641, 2653, -1), type="misc_feature", qualifiers = {'label': 'U13'})
        f14 = SeqFeature(FeatureLocation(2653, 3056, -1), type="misc_feature", qualifiers = {'label': 'Human PolI promoter'})
        f15 = SeqFeature(FeatureLocation(3923, 4781, -1), type="CDS", qualifiers = {'label': 'AmpR'})

    elif subtype=='H1N1':
        f4 = SeqFeature(FeatureLocation(732, 792, +1), type="misc_feature", qualifiers = {'label': 'WSN first 20 aa'})
        f5 = SeqFeature(FeatureLocation(792, 2292, +1), type="misc_feature", qualifiers = {'label': f'HA gene from {strain_name}'})
        f6 = SeqFeature(FeatureLocation(2292, 2436, +1), type="misc_feature", qualifiers = {'label': 'WSN endodomain'})
        f8 = SeqFeature(FeatureLocation(2403, 2436, +1), type="misc_feature", qualifiers = {'label': 'WSN recoded CT'})
        f9 = SeqFeature(FeatureLocation(2436, 2452, +1), type="misc_feature", qualifiers = {'label': 'barcode'})
        f10 = SeqFeature(FeatureLocation(2452, 2485, +1), type="misc_feature", qualifiers = {'label': 'Illumina Read1'})
        f11 = SeqFeature(FeatureLocation(2494, 2599, +1), type="misc_feature", qualifiers = {'label': 'WSN packaging signal'})
        f12 = SeqFeature(FeatureLocation(2599, 2644, -1), type="misc_feature", qualifiers = {'label': "5' NCR"})
        f13 = SeqFeature(FeatureLocation(2632, 2644, -1), type="misc_feature", qualifiers = {'label': 'U13'})
        f14 = SeqFeature(FeatureLocation(2644, 3047, -1), type="misc_feature", qualifiers = {'label': 'Human PolI promoter'})
        f15 = SeqFeature(FeatureLocation(3914, 4772, -1), type="CDS", qualifiers = {'label': 'AmpR'})

    features_list = [f1,f2,f3,f4,f5,f6,f8,f9,f10,f11,f12,f13,f14,f15]
    
    # Write sequence record and save 
    record = SeqRecord(Seq(plasmid_sequence), 
                       id = '.', 
                       name = plasmid_name, description = definition, 
                       features = features_list, 
                       annotations = {'source': 'synthetic DNA construct',
                                      'organism': 'synthetic DNA construct',
                                      'molecule_type': 'ds-DNA',
                                      'topology': 'circular',
                                      'date': '4-SEP-2025'})
    
    return(plasmid_sequence, definition, record)    
    

In [23]:
# Write records for all plasmids

for plasmid in final_inserts_df.plasmid.unique():
    temp_df = final_inserts_df.query(f'plasmid == "{plasmid}"')

    plasmid_sequence, defintion, record = (write_genbank(
        strain_name = temp_df.strain.values[0],
        accession = temp_df.genbank.values[0],
        plasmid_name = temp_df.plasmid.values[0],
        subtype = temp_df.subtype.values[0],
        ectodomain = temp_df.trimmed_ha_ecto_sequence.values[0],
        barcode = temp_df.barcode.values[0],
        seqneut_library = 'flu-seqneut-2025'
        ))

    logname = temp_df.plate_ID.values[0] + '_' + temp_df.plasmid.values[0]
    outfile = os.path.join(genbankdir, f'{logname}.gb')
    with open(outfile, 'w') as f:
        SeqIO.write(record, f, 'genbank')


In [24]:
# Save output with plasmid log IDs
final_inserts_df['bloom_plasmid_log_name'] = final_inserts_df['plate_ID'] + '_' + final_inserts_df['plasmid']
final_inserts_df.to_csv(os.path.join(resultsdir, 'summary_tables', 'constructs_to_plasmid_log_id_mapping.csv'), index=False)

/tmp/ipykernel_61847/3152475070.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_inserts_df['bloom_plasmid_log_name'] = final_inserts_df['plate_ID'] + '_' + final_inserts_df['plasmid']
